# Iridium-1 — train the routed control-core model

One control core that every token passes through, dispatching to deep superstacks,
choosing its own depth and how many passes to spend. Omnimodal in and out:
text, image, video, audio, physical fields, geometry, actions, and typed quantities.

**Runtime → Change runtime type → GPU**, then **Runtime → Run all**.

| rung | params | what a GPU needs | time |
|---|---|---|---|
| `nano` | 34 M | anything | ~5 min |
| `nano100m` | 104 M | anything | ~12 min |
| `micro` | 1.0 B | 16 GB (T4 works) | ~30 min |
| `test1b` | 1.0 B | 24 GB preferred | ~40 min |

> **T4 caveat.** A free Colab T4 is Turing and has no bf16. This notebook detects that
> and runs fp32 rather than fp16, because the router's softmax and the attention
> logits are exactly where fp16 overflows — and a collapsed router looks like a bad
> run rather than a numerics bug. On an A100/L4 it uses bf16 and is much faster.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv || echo 'NO GPU — set Runtime > Change runtime type > GPU'


## 1 · Get the code


In [ ]:
!git clone --depth 1 --branch claude/gallant-faraday-lhycva https://github.com/sporadicstudiosind-cloud/test.git iridium 2>/dev/null || (cd iridium && git pull)
%cd iridium
!pip -q install pyyaml
import sys; sys.path.insert(0, '.')
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2 · Pick the rung

The same modules serve every rung — scale is a config file, not a rewrite.
`iridium ladder` costs all of them, including the ones nobody can build.


In [ ]:
!python -m iridium ladder


In [ ]:
RUNG   = 'nano100m'   # 'nano' 34M | 'nano100m' 104M | 'micro'/'test1b' 1.0B
STEPS  = 4000
BATCH  = 64
LR     = 5e-4
# Full omnimodal mixture. Narrow it to train faster on a weak GPU.
MIXTURE = ('channel_depth=0.30,channel_intervention=0.25,false_premise=0.15,'
           'field_rollout=0.15,scene_goal=0.15')

from iridium.config import get_config
cfg = get_config(RUNG)
print(cfg.report().render())


## 3 · Check the architecture before spending a GPU hour on it

The parameter formulae that cost a 9-trillion-parameter configuration are the same
ones that describe this model. At the rungs that instantiate, the difference between
formula and modules is exactly zero — and the cache-parity gate proves that cached
decoding computes what teacher forcing trained. If either fails, stop.


In [ ]:
!python -m pytest tests/unit/test_config_inventory.py tests/integration/test_kv_parity.py -q


## 4 · Train


In [ ]:
import torch, time, json
from pathlib import Path
from iridium.model.iridium1 import Iridium1
from iridium.training.datasets import build_corpus, describe
from iridium.training.trainer import TrainConfig, Trainer
from iridium.training.losses import LossWeights

device = 'cuda' if torch.cuda.is_available() else 'cpu'
major = torch.cuda.get_device_capability()[0] if device == 'cuda' else 0
# Ampere (8.x) and newer have bf16. Turing (7.5, the free T4) does not, and fp16
# is the wrong trade for a routed model: see the caveat at the top.
use_bf16 = device == 'cuda' and major >= 8
print(f'device={device} compute_capability={major}.x bf16={use_bf16}')

mixture = {k: float(v) for k, v in (p.split('=') for p in MIXTURE.split(','))}
train = build_corpus(40000, seed=0, split='train', mixture=mixture)
test  = build_corpus(800, seed=1000, split='test', mixture=mixture)
extra = build_corpus(400, seed=2000, split='extrapolation', mixture=mixture)
print(describe(train))

model = Iridium1(cfg).to(device)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters on {device}')


In [ ]:
from iridium.evaluation.harness import evaluate

before = evaluate(model, test, max_per_family=16)
print('untrained:', json.dumps(before, indent=1)[:900])


In [ ]:
tcfg = TrainConfig(steps=STEPS, batch_size=BATCH, lr=LR, seed=0,
                   label=f'colab-{RUNG}', log_every=max(STEPS//50, 1),
                   checkpoint_every=max(STEPS//5, 1))
trainer = Trainer(model, train, tcfg, LossWeights(), out_dir=Path('runs/colab'),
                  device=device)
t0 = time.time()
history = trainer.train()
print(f'trained in {(time.time()-t0)/60:.1f} min')


## 5 · Grade it

Not loss — **graded accuracy**, from free-running generation, checked against an
independent computation: the analytic Manning law, the spectral solver, or the scene
environment's own goal predicate. Every score is reported beside the baseline a model
gets by ignoring its input entirely, because a number without a baseline cannot be read.

`extrapolation` draws discharges from a band the training split never contains, so a
score there cannot be earned by having seen a neighbouring example.


In [ ]:
model.eval()
results = {
    'interpolation': evaluate(model, test, max_per_family=48),
    'extrapolation': evaluate(model, extra, max_per_family=48),
    'untrained_baseline': before,
}
print(json.dumps(results, indent=2))

print()
print(f"{'family':<26}{'trained':>9}{'baseline':>10}{'verdict':>14}")
for fam, r in results['interpolation'].items():
    acc, base = r.get('accuracy', 0), r.get('baseline', 0)
    verdict = 'LEARNED' if acc > base + 0.1 else ('at baseline' if acc >= base else 'BELOW baseline')
    print(f'{fam:<26}{acc:>9.3f}{base:>10.3f}{verdict:>14}')


## 6 · Look at the routing

Balance and specialisation pull in opposite directions, and a router can look healthy
on either while failing the other. `I(family; stack)` separates them: zero means routing
is independent of the task, `log(min(families, stacks))` means a clean partition.


In [ ]:
from iridium.evaluation.routing import analyse
from iridium.training.datasets import BatchLoader
report = analyse(model, BatchLoader(test, cfg.codecs, 16, 0, device=device))
print(report.render())


## 7 · Save the weights

Stored fp16 so the file stays small. Mount Drive to keep it past the session.


In [ ]:
path = trainer.save('final', extra={'evaluation': results})
print('saved', path)

import torch
blob = torch.load(path, map_location='cpu', weights_only=False)
half = {k: (v.half() if v.is_floating_point() else v) for k, v in blob['state_dict'].items()}
torch.save({'state_dict': half, 'manifest': blob['manifest']}, 'iridium-fp16.pt')
print('fp16 checkpoint written')

# from google.colab import drive; drive.mount('/content/drive')
# !cp iridium-fp16.pt /content/drive/MyDrive/

from google.colab import files
files.download('iridium-fp16.pt')


## 8 · Talk to it

Ask for a number and it answers with one — beside the analytic value, so you can check
it rather than believe it. Numbers travel as typed quantities, never as decimal prose:
the same mapping learned from digit-bytes reaches 18.6% of answers inside a 2% tolerance,
and learned from typed values, 100%.


In [ ]:
import subprocess, os, threading
os.environ['IRIDIUM_CHECKPOINT'] = str(path)
os.environ['PYTHONPATH'] = '.'
os.environ['PORT'] = '8080'
threading.Thread(target=lambda: subprocess.run(['python','serve/server.py']), daemon=True).start()
import time; time.sleep(25)

import json, urllib.request
def ask(q, loops=1):
    body = json.dumps({'prompt': q, 'loops': loops}).encode()
    req = urllib.request.Request('http://127.0.0.1:8080/api/ask', body,
                                 {'Content-Type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=120))

for q in ['normal depth | S=0.0020 n=0.030 q=3.0',
          'depth ratio | S=0.0020 n=0.030 q=3.0 x2.0',
          'doubling the discharge doubles the flow depth']:
    r = ask(q)
    a = r['answer']
    if 'predicted' in a:
        print(f"{q}\n   model {a['predicted']:.4f}  analytic {a['analytic']:.4f}"
              f"  rel err {a['relative_error']:.4f}  within 2%: {a['within_2pct']}")
    else:
        print(f"{q}\n   verdict {a['verdict']}")
    t = r['telemetry']
    print(f"   routed to {t['stacks_used']}/{len(t['routing'])} stacks,"
          f" focus {t['mean_focus']:.3f}, {t['expected_loops']:.2f} ponder loops")


In [ ]:
from google.colab.output import eval_js
print('Probe UI:', eval_js('google.colab.kernel.proxyPort(8080)'))


---

### What a good run looks like

- `channel_depth` and `channel_intervention` **above 0.9** on interpolation. These are
  affine in log space once the inputs arrive as typed quantities, so anything much
  below that means something is wrong, not that the task is hard.
- `extrapolation` will be **lower** — that band is outside the training discharges, and
  the gap between the two columns is the honest measure of what was learned versus fitted.
- `false_premise` draws from only ten distinct claims, so a high score there is
  **memorisation, not judgement**. Do not read it as calibration.
- `field_rollout` must beat the persistence baseline (emit the input frame unchanged)
  to mean anything at all.
- Stack-usage entropy near `log(n_stacks)` means no collapse; `I(family; stack)` near
  zero means no specialisation yet — that is what phase 2 is for.
